# CAFA 6 - Generate Submission

Load trained models, predict on test set, apply GO propagation, validate, and save.

In [ ]:
# === Google Colab Setup (skip if running locally) ===
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/AyushSharma173/CafaChallenge.git
    %cd CafaChallenge
    !pip install -q obonet biopython h5py lightgbm scikit-learn matplotlib seaborn pyyaml
    
    # Kaggle API auth - upload your kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        from google.colab import files
        print('Upload your kaggle.json (get it from https://kaggle.com/settings > API > Create New Token)')
        uploaded = files.upload()
        !mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    
    !kaggle competitions download -c cafa-6-protein-function-prediction -p data/raw
    !unzip -qo data/raw/cafa-6-protein-function-prediction.zip -d data/raw/
    %cd notebooks
    print('Colab setup complete!')

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from cafa6.config import Config
from cafa6.data_loader import load_fasta
from cafa6.embeddings import load_embeddings
from cafa6.go_utils import load_go_graph
from cafa6.models import BaseModel
from cafa6.submission import (
    generate_submission, validate_submission, predictions_from_matrices
)

cfg = Config.from_yaml('../configs/default.yaml')
DATA_DIR = Path('../data/raw')

MODEL_TYPE = 'lightgbm'  # or 'mlp'

## 1. Load Test Embeddings & Models

In [ ]:
# Load test embeddings
emb_file = Path(cfg.embeddings_dir) / f'test_{cfg.embeddings.model_name}.h5'
test_embs, test_ids = load_embeddings(emb_file)
print(f'Test embeddings: {test_embs.shape}')

# Load GO graph
go_graph = load_go_graph(DATA_DIR / 'Train' / 'go-basic.obo')

## 2. Generate Predictions

In [ ]:
score_matrices = {}

for aspect in cfg.ontologies:
    model = BaseModel.load(f'../models/{MODEL_TYPE}_{aspect}.pkl')
    term_ids = list(np.load(f'../models/term_ids_{aspect}.npy', allow_pickle=True))
    
    scores = model.predict(test_embs.astype(np.float32))
    score_matrices[aspect] = (scores, test_ids, term_ids)
    
    print(f'{aspect}: {scores.shape}, '
          f'mean_conf={scores.mean():.4f}, max_conf={scores.max():.4f}')

## 3. Convert to Submission & Propagate

In [ ]:
predictions = predictions_from_matrices(score_matrices)
print(f'Predictions for {len(predictions)} proteins')

# Stats before propagation
n_preds_before = sum(len(v) for v in predictions.values())
print(f'Total (protein, term) pairs before propagation: {n_preds_before:,}')

In [ ]:
output_path = f'../submissions/{MODEL_TYPE}_submission.csv'

sub_df = generate_submission(
    predictions,
    go_graph=go_graph,
    output_path=output_path,
    propagate=True,
    min_confidence=cfg.min_confidence
)

## 4. Validate

In [ ]:
test_fasta = load_fasta(DATA_DIR / 'Test' / 'testsuperset.fasta')
issues = validate_submission(
    output_path,
    test_protein_ids=set(test_fasta.keys()),
    valid_go_terms=set(go_graph.nodes())
)

if not issues:
    print('\nReady to submit to Kaggle!')
    print(f'File: {output_path}')

## 5. Submission Statistics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence distribution
axes[0].hist(sub_df['confidence'], bins=100, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence Score Distribution')

# Predictions per protein
preds_per_protein = sub_df.groupby('protein_id').size()
axes[1].hist(preds_per_protein, bins=100, color='coral', alpha=0.7)
axes[1].set_xlabel('Predictions per Protein')
axes[1].set_ylabel('Count')
axes[1].set_title('Predictions per Protein')

plt.tight_layout()
plt.show()

print(f'Submission shape: {sub_df.shape}')
print(f'\nConfidence stats:\n{sub_df["confidence"].describe()}')
print(f'\nPredictions per protein:\n{preds_per_protein.describe()}')

## 6. Submit to Kaggle (Optional)

```bash
kaggle competitions submit -c cafa-6-protein-function-prediction \
    -f submissions/lightgbm_submission.csv \
    -m "LightGBM on ESM-2 650M embeddings"
```